In [1]:
import pandas as pd
import heapq
import time

# Load cleaned dataset
df = pd.read_csv("../data/processed/edges_clean.csv")

# Build graph
graph = {}
for _, row in df.iterrows():
    graph.setdefault(int(row["from_node"]), {})[int(row["to_node"])] = int(row["cost"])

graph


{1: {2: 15, 5: 19, 6: 11},
 2: {3: 2, 6: 1, 7: 12, 8: 10},
 3: {1: 16, 2: 15, 5: 3},
 4: {1: 14, 3: 2, 7: 8, 8: 14},
 5: {1: 18, 3: 6, 4: 4, 6: 18},
 6: {4: 8},
 7: {4: 15, 5: 7, 8: 8},
 8: {1: 11, 2: 8, 5: 7}}

In [2]:
def greedy_route(graph, start):
    visited = {start}
    route = [start]
    total_cost = 0
    current = start

    while len(visited) < len(graph):
        neighbors = graph.get(current, {})
        next_node = None
        min_cost = float("inf")

        for node, cost in neighbors.items():
            if node not in visited and cost < min_cost:
                next_node = node
                min_cost = cost

        if next_node is None:
            break

        route.append(next_node)
        total_cost += min_cost
        visited.add(next_node)
        current = next_node

    return route, total_cost


In [3]:
def dijkstra_with_path(graph, start):
    dist = {node: float("inf") for node in graph}
    prev = {node: None for node in graph}

    dist[start] = 0
    pq = [(0, start)]

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        if current_dist > dist[current_node]:
            continue

        for neighbor, cost in graph.get(current_node, {}).items():
            new_dist = current_dist + cost
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                prev[neighbor] = current_node
                heapq.heappush(pq, (new_dist, neighbor))

    return dist, prev


In [4]:
def heuristic(node, goal):
    return abs(node - goal)

def astar(graph, start, goal):
    open_set = [(0, start)]
    g_cost = {start: 0}
    prev = {start: None}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            break

        for neighbor, cost in graph.get(current, {}).items():
            tentative = g_cost[current] + cost
            if neighbor not in g_cost or tentative < g_cost[neighbor]:
                g_cost[neighbor] = tentative
                f_cost = tentative + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_cost, neighbor))
                prev[neighbor] = current

    return g_cost, prev


In [5]:
def reconstruct_path(prev, start, target):
    path = []
    current = target
    while current is not None:
        path.append(current)
        current = prev.get(current)
    path.reverse()
    return path if path[0] == start else []


In [6]:
start_node = 1
target_node = 5

# Greedy
t0 = time.time()
g_route, g_cost = greedy_route(graph, start_node)
t1 = time.time()

# Dijkstra
t2 = time.time()
d_dist, d_prev = dijkstra_with_path(graph, start_node)
d_path = reconstruct_path(d_prev, start_node, target_node)
t3 = time.time()

# A*
t4 = time.time()
a_costs, a_prev = astar(graph, start_node, target_node)
a_path = reconstruct_path(a_prev, start_node, target_node)
t5 = time.time()


In [7]:
results = pd.DataFrame({
    "Algorithm": ["Greedy", "Dijkstra", "A*"],
    "Path": [g_route, d_path, a_path],
    "Total Cost": [g_cost, d_dist[target_node], a_costs[target_node]],
    "Execution Time (ms)": [
        (t1 - t0) * 1000,
        (t3 - t2) * 1000,
        (t5 - t4) * 1000
    ]
})

results


,Algorithm,Path,Total Cost,Execution Time (ms)
0,Greedy,"[1, 6, 4, 3, 5]",24,0.000000
1,Dijkstra,"[1, 5]",19,1.511812
2,A*,"[1, 5]",19,0.000000
